# <span style="color: purple;">Lesson 11 - Feature Engineering</span>

**Developers:** Brian  |  **Reviewers:** 

---

## <span style="color: purple;">🎯 Learning Objectives</span>

By the end of this lesson, you will have:

- Learned how transforming and creating features can improve model performance.
- Transformed an existing feature to reveal clearer patterns.
- Created a new variable by combining existing ones.
- Evaluated performance changes before and after feature engineering.

<div class="alert alert-info">

**How it works:** Click the ▶ button on the top left of each code cell, top to bottom. Read the explanation after you see the output. That's it!

</div>

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup cell — run this first, then you can ignore it
# ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from tqdm.notebook import tqdm

print("✓ Ready to go!")

---

## <span style="color: purple;">🧩 What is Feature Engineering?</span>

Before deep learning and massive neural networks became popular, machine learning models were relatively simple algorithms. Because these early models couldn't automatically figure out complex, hidden relationships in the data on their own, human experts had to mathematically craft the perfect input variables for them. 

Historically, this process of solving inference problems by carefully crafting data was the absolute secret to winning early machine learning competitions. Data scientists would spend 90% of their time just manipulating columns of data before a model even saw it!

Even today, for many tabular (spreadsheet) datasets, giving a model "better" and more explicit information is often far more effective—and much faster—than just building a bigger, more complicated AI brain. A model is truly only as good as the data it is trained on.

<div class="alert alert-success">

**Feature Engineering** - The art and science of creating new features or transforming existing ones to explicitly give a model "better" information so it can make better, easier predictions.

</div>

### Why do we need it? 
Sometimes, the raw data we are given holds the truth, but it's hidden in disconnected pieces. 

Imagine you are building an mmodel to predict if someone will need a winter coat outside. 
- You give the model their **Body Temperature** and the **Outside Temperature**. 
- A simple model might struggle to figure out how these two independent numbers relate to freezing. It just sees them as two separate columns.
- But if you engineer a new feature called **Temperature Difference** (by subtracting Outside Temp from Body Temp), you are instantly handing the model the exact, unified piece of information it needs to solve the puzzle perfectly!

This involves two main actions:
- **Transforming** numbers so they are easier for a model to digest (e.g., changing wild, skewed population numbers into a smoother bell-curve scale using logarithms).
- **Creating new variables** out of old ones (e.g., calculating 'Speed' mathematically from 'Distance' divided by 'Time' when predicting how dangerous a car crash might be).

---

## <span style="color: purple;">Step 1 - Look at the data</span>

We will look at a dataset of California districts, trying to predict the **Average House Value** in each district based on the local features.

<div class="alert class-success">

**Regression** - Since we are predicting a continuous number (house price) rather than sorting into categories (like cat or dog), this is called a Regression task!

</div>

Let's load our data and check the first few rows.

<div class="alert alert-info">

**Run the next cell** to load the data and view a table.

</div>

In [ ]:
# Load housing dataset
data = fetch_california_housing(as_frame=True)
df = data.frame

# Make the feature names a bit clearer
df = df.rename(columns={
    'MedInc': 'Median_Income',
    'HouseAge': 'House_Age',
    'AveRooms': 'Average_Rooms',
    'Population': 'Total_Population',
    'AveOccup': 'Average_Occupancy',
    'MedHouseVal': 'House_Value'
})

display(df.head())
print("✓ Data loaded")

<div class="alert alert-info">

### 👀 The Raw Data

Notice the features like `Average_Rooms` (total rooms divided by households in a district) and `Average_Occupancy` (people per household).

Let's train a model on this **raw** dataset to see our starting performance.

</div>

---

## <span style="color: purple;">Step 2 - Train a baseline model</span>

We will use a Decision Tree to predict `House_Value`, and measure performance using Mean Absolute Error (how many hundred-thousand dollars off our price guesses are, on average).

In [ ]:
# Set our starting features and target
X_raw = df[['Median_Income', 'House_Age', 'Average_Rooms', 'Total_Population', 'Average_Occupancy']]
y_target = df['House_Value']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_target, test_size=0.2, random_state=42)

# Train a simple Decision Tree Regressor
baseline_model = DecisionTreeRegressor(max_depth=5, random_state=42)
baseline_model.fit(X_train, y_train)

# Test it
predictions = baseline_model.predict(X_test)
baseline_error = mean_absolute_error(y_test, predictions)

print(f"✓ Baseline Mean Absolute Error: {baseline_error:.3f} (Roughly ${baseline_error * 100000:,.0f} off per guess)")

<div class="alert alert-info">

Our baseline model is off by about $61,000 per guess. Let's see if we can do better by engineering some new features!

</div>

---

## <span style="color: purple;">Step 3 - Create new variables</span>

Instead of just looking at `Average_Rooms` and `Average_Occupancy` separately, what if we made a new feature called `Rooms_Per_Person`? A house might have many rooms, but if it has a huge occupancy, the rooms-per-person is tiny. This might strongly influence house values!

<div class="alert alert-success">

**Creating New Variables** - Mathematically combining multiple existing columns into one new column that carries a clearer, stronger meaning.

</div>

In [ ]:
# Create a new feature!
df['Rooms_Per_Person'] = df['Average_Rooms'] / df['Average_Occupancy']

display(df[['Average_Rooms', 'Average_Occupancy', 'Rooms_Per_Person', 'House_Value']].head())
print("✓ New variable created")

---

## <span style="color: purple;">Step 4 - Transform existing features</span>

Some numbers simply span too large a range. Look at `Total_Population`. Most districts might have 1,000 people, but a few have 30,000! Models often struggle with variables that skew so extremely.

We can **transform** it by taking the logarithm (squashing giant numbers down so the model doesn't get distracted by them).

<div class="alert alert-success">

**Transforming Features** - Changing the scale or shape of data (like squishing extreme values) so the model isn't confused by outliers.

</div>

In [ ]:
# Transform: squish the giant population numbers
df['Log_Population'] = np.log1p(df['Total_Population'])

# Let's see the before & after visually
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df['Total_Population'].hist(bins=30)
plt.title("Original Population (Skewed)")

plt.subplot(1, 2, 2)
df['Log_Population'].hist(bins=30, color='orange')
plt.title("Transformed Population (Log)")
plt.show()

print("✓ Feature transformed")

<div class="alert alert-info">

Notice how the original population graph has a long, microscopic tail trailing off to the right, whereas the orange graph looks much more like a neat bell curve. Models *love* neat bell curves!

</div>

---

## <span style="color: purple;">Step 5 - Evaluate performance changes</span>

Let's retrain our exact same model, but this time we'll give it our newly crafted features to see if its error goes down.

In [ ]:
# Update our features to include our new ones, and swap Population for Log_Population!
X_engineered = df[['Median_Income', 'House_Age', 'Average_Rooms', 'Log_Population', 'Average_Occupancy', 'Rooms_Per_Person']]

X_train_eng, X_test_eng, y_train, y_test = train_test_split(X_engineered, y_target, test_size=0.2, random_state=42)

# Train the exact same type of model
engineered_model = DecisionTreeRegressor(max_depth=5, random_state=42)
engineered_model.fit(X_train_eng, y_train)

eng_predictions = engineered_model.predict(X_test_eng)
engineered_error = mean_absolute_error(y_test, eng_predictions)

print(f"Old Error (Baseline):   {baseline_error:.3f}")
print(f"New Error (Engineered): {engineered_error:.3f}")

if engineered_error < baseline_error:
    print(f"✓ Success! We improved the model just by changing the data!")
else:
    print("Oops, our features didn't help!")

---

## <span style="color: purple;">🧠 What did we just do?</span>

We noticed that our raw data was good, but could be presented to the model in a "cleaner" way.

By calculating the **ratio** of rooms to people, we gave the model a concept it couldn't easily figure out. By **transforming** population onto a logarithmic scale, we stopped extreme outliers from confusing the model. 

Most importantly, our model's predictions improved **without changing the model itself**. Sometimes better data is more valuable than a deeper AI!

---

### Key words

| Word | What it means |
|------|--------------|
| **Feature Engineering** | Creating new features or transforming existing ones to improve models |
| **New Variables** | Combining multiple columns mathematically (e.g., A/B) |
| **Transforming Features** | Adjusting the shape/scale of a column so outliers are managed |
| **Log Transform** | A mathematical trick used to compress huge numbers into a simpler scale |

---

## <span style="color: purple;">🔧 Your turn!</span>

Can you create another new variable? Does `Median_Income` multiplied by `House_Age` mean anything? Try making a new feature `Income_Age_Combo` and add it to `X_engineered`!

**Change the code below and run it to see if you can lower the error even further!**

In [ ]:
# ▼▼▼▼▼▼▼▼▼▼▼▼▼▼  CHANGE THIS  ▼▼▼▼▼▼▼▼▼▼▼▼▼▼
# Try to create your own new feature math here:
df['My_New_Feature'] = df['Total_Population'] + df['Average_Rooms']   # ← try multiplying or dividing things instead
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲  CHANGE THIS  ▲▲▲▲▲▲▲▲▲▲▲▲▲▲

# Update our features
X_challenge = df[['Median_Income', 'House_Age', 'Average_Rooms', 'Log_Population', 'Average_Occupancy', 'Rooms_Per_Person', 'My_New_Feature']]

X_train_c, X_test_c, y_train, y_test = train_test_split(X_challenge, y_target, test_size=0.2, random_state=42)

challenge_model = DecisionTreeRegressor(max_depth=5, random_state=42)
challenge_model.fit(X_train_c, y_train)

ch_predictions = challenge_model.predict(X_test_c)
ch_error = mean_absolute_error(y_test, ch_predictions)

print(f"Previous Engineered Error: {engineered_error:.4f}")
print(f"Your Challenge Error:      {ch_error:.4f}")
print("✓ Challenge completed")

---

## <span style="color: purple;">🧪 Quick Quiz</span>

Test yourself on what you just learned. Run the code cell, select your answer for each question, then click **Check answers**.

In [ ]:
from quiz import run_b11_quiz
run_b11_quiz()